# Benchmark Convergence — v1LD (long terme)

Compare la convergence à long terme des deux variantes du solveur sous-gradient :
- **DP exact** (`decomp=False`) : ζ_i résolu par bitmask DP entier  
- **DP décomposé** (`decomp=True`) : ζ̂_i via décomposition interne μ  

Critères d'arrêt (configurables) :
- limite de temps (`TIME_LIMIT_S`)
- amélioration relative < `REL_TOL` sur fenêtre de `TOL_WINDOW` itérations

Une figure JPG est sauvegardée par instance dans `figures/convergence/`.

In [1]:
import sys, os, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, '.')
import v1ld_cpp2
from solver.instance_reader import load_instances
from solver.mip_reference import solve_mip_exact

# ── Configuration ─────────────────────────────────────────────────────────────
SIZES        = ["small", "medium"]   # tailles d'instances à tester
N_EPISODES   = 5                      # épisodes par taille
TIME_LIMIT_S = 20                    # secondes max par solveur par instance
                                      # (mettre 1800 pour 30 min)
LOG_INTERVAL = 50                     # enregistrer 1 point toutes les N itérations
T_MAX        = 10_000_000             # borne supérieure (rarement atteinte)
REL_TOL      = 1e-6                   # seuil d'amélioration relative (arrêt)
TOL_WINDOW   = 1000                   # fenêtre d'itérations pour REL_TOL
FIGURE_DIR   = 'figures/convergence'

os.makedirs(FIGURE_DIR, exist_ok=True)
print(f"Config : TIME_LIMIT={TIME_LIMIT_S}s, LOG_INTERVAL={LOG_INTERVAL}, "
      f"REL_TOL={REL_TOL}, TOL_WINDOW={TOL_WINDOW}")
print(f"Figures sauvegardées dans : {FIGURE_DIR}/")

Config : TIME_LIMIT=20s, LOG_INTERVAL=50, REL_TOL=1e-06, TOL_WINDOW=1000
Figures sauvegardées dans : figures/convergence/


In [2]:
# ── Chargement des instances, références MIP et v_C(M) ───────────────────────
from solver.v1ld_solver_general import compute_vc_m

DATA = {}
for size in SIZES:
    path  = f"data/covset/datasets/test/ccsp-test-{size}.txt"
    insts = load_instances(path, epsilon=0.1, max_episodes=N_EPISODES)
    refs  = {}
    print(f"\n── {size.upper()} ({len(insts)} épisodes) ──")
    for inst in insts:
        v_mip, status = solve_mip_exact(inst)
        vcm           = compute_vc_m(inst)
        refs[inst.episode] = {
            'instance':   inst,
            'v_mip':      v_mip,
            'v_CM_exact': vcm['v_CM_exact'],  # = v_mip pour A,b ∈ {0,1}
            'v_CM_lp':    vcm['v_CM_lp'],     # LP complète (x et z continus)
            'z_lp_frac':  vcm['z_lp_frac'],
        }
        print(f"  ep={inst.episode}: n={inst.n}, m={inst.m}, N={inst.N}  "
              f"v*={v_mip:.3f} [{status}]  "
              f"v_C(M)_LP={vcm['v_CM_lp']:.3f}  "
              f"v_C(M)_exact={vcm['v_CM_exact']:.3f}  "
              f"z_frac={vcm['z_lp_frac']}")
    DATA[size] = refs

print("\nInstances, MIP et v_C(M) chargées.")
print("Note : v_C(M)_exact = v* pour les instances binaires (relaxer z seul n'apporte rien).")
print("       v_C(M)_LP    = relaxation LP complète (borne inférieure utile).")



── SMALL (5 épisodes) ──
  ep=0: n=20, m=5, N=5  v*=4.000 [Optimal]  v_C(M)_LP=3.000  v_C(M)_exact=4.000  z_frac=[(0, 0.4999999999999999)]
  ep=1: n=20, m=5, N=5  v*=4.000 [Optimal]  v_C(M)_LP=3.500  v_C(M)_exact=4.000  z_frac=[(0, 0.4999999999999996)]
  ep=2: n=20, m=5, N=5  v*=3.000 [Optimal]  v_C(M)_LP=2.000  v_C(M)_exact=3.000  z_frac=[(2, 0.5)]
  ep=3: n=20, m=5, N=5  v*=3.000 [Optimal]  v_C(M)_LP=2.042  v_C(M)_exact=3.000  z_frac=[(2, 0.708333333333333), (4, 0.7916666666666666)]
  ep=4: n=20, m=5, N=5  v*=4.000 [Optimal]  v_C(M)_LP=3.125  v_C(M)_exact=4.000  z_frac=[(4, 0.4999999999999999)]

── MEDIUM (5 épisodes) ──
  ep=0: n=50, m=5, N=10  v*=3.000 [Optimal]  v_C(M)_LP=2.778  v_C(M)_exact=3.000  z_frac=[(0, 0.833333333333332), (1, 0.7777777777777803), (4, 0.8333333333333298), (8, 0.6111111111111136), (9, 0.9444444444444443)]
  ep=1: n=50, m=5, N=10  v*=3.000 [Optimal]  v_C(M)_LP=1.681  v_C(M)_exact=3.000  z_frac=[(0, 0.9064449064448987), (3, 0.7494802494802689), (4, 0.94802494

In [3]:
# ── Fonction de lancement d'un solveur avec critères d'arrêt ──────────────────
def run_solver(inst, decomp):
    res = v1ld_cpp2.solve(
        inst.c.astype(float), inst.p.astype(float),
        inst.A.astype(float), inst.b.astype(float),
        inst.epsilon,
        T=T_MAX,
        decomp=decomp,
        verbose=False,
        time_limit_s=TIME_LIMIT_S,
        log_interval=LOG_INTERVAL,
        rel_tol=REL_TOL,
        tol_window=TOL_WINDOW,
    )
    conv  = np.array(res['convergence'])
    times = np.array(res['times_s'])
    best  = np.maximum.accumulate(conv) if len(conv) > 0 else conv
    return {
        'conv':               conv,
        'best':               best,
        'times_s':            times,
        'v1LD_star':          res['v1LD_star'],
        'iters':              res['iterations_run'],
        'wall_s':             res['solve_time_s'],
        'g_history':          [np.array(g) for g in res.get('g_history', [])],
        'xbar_outer_history': [np.array(x) for x in res.get('xbar_outer_history', [])],
        'zstar_history':      [np.array(z) for z in res.get('zstar_history', [])],
        'iter_history':       list(res.get('iter_history', [])),
        'lambda_star':        np.array(res.get('lambda_star', [])),
    }

# ── Exécution sur toutes les instances ────────────────────────────────────────
RESULTS = {}
for size in SIZES:
    RESULTS[size] = {}
    for ep in sorted(DATA[size]):
        inst = DATA[size][ep]['instance']

        print(f"[{size} ep={ep}] exact ... ", end='', flush=True)
        r_exact  = run_solver(inst, decomp=False)
        print(f"{r_exact['iters']:,} iters, {r_exact['wall_s']:.1f}s  "
              f"→ v1LD*={r_exact['v1LD_star']:.4f}")

        print(f"[{size} ep={ep}] decomp .. ", end='', flush=True)
        r_decomp = run_solver(inst, decomp=True)
        print(f"{r_decomp['iters']:,} iters, {r_decomp['wall_s']:.1f}s  "
              f"→ v1LD*={r_decomp['v1LD_star']:.4f}")

        RESULTS[size][ep] = {'exact': r_exact, 'decomp': r_decomp}

print("\nCalculs terminés.")

[small ep=0] exact ... 8,001 iters, 0.0s  → v1LD*=3.9911
[small ep=0] decomp .. 98,001 iters, 1.7s  → v1LD*=3.6842
[small ep=1] exact ... 2,001 iters, 0.0s  → v1LD*=4.0000
[small ep=1] decomp .. 31,001 iters, 0.6s  → v1LD*=4.0000
[small ep=2] exact ... 9,001 iters, 0.0s  → v1LD*=2.3115
[small ep=2] decomp .. 254,001 iters, 3.8s  → v1LD*=2.2884
[small ep=3] exact ... 6,001 iters, 0.0s  → v1LD*=2.4084
[small ep=3] decomp .. 57,001 iters, 0.9s  → v1LD*=2.3927
[small ep=4] exact ... 2,001 iters, 0.0s  → v1LD*=4.0000
[small ep=4] decomp .. 37,001 iters, 0.6s  → v1LD*=4.0000
[medium ep=0] exact ... 4,001 iters, 0.0s  → v1LD*=2.7129
[medium ep=0] decomp .. 209,001 iters, 6.9s  → v1LD*=2.6590
[medium ep=1] exact ... 7,001 iters, 0.1s  → v1LD*=1.6450
[medium ep=1] decomp .. 94,001 iters, 3.1s  → v1LD*=1.5447
[medium ep=2] exact ... 9,001 iters, 0.1s  → v1LD*=2.6679
[medium ep=2] decomp .. 114,001 iters, 3.8s  → v1LD*=2.5652
[medium ep=3] exact ... 7,001 iters, 0.1s  → v1LD*=1.4448
[medium ep=3]

In [4]:
# ── Affichage des 5 derniers snapshots (multiples de 5) pour chaque instance ──
#
# Pour chaque instance et chaque variante (exact / decomp) on affiche,
# aux 5 dernières itérations multiples de 5 :
#   - le numéro d'itération t
#   - la norme ||g_λ||₂  (gradient par rapport à λ)
#   - la norme ||x̄_outer||₂
#   - z* (solution du knapsack, vecteur 0/1)
#   - x_ζ_i pour chaque scénario i  (x_i* = g[i] + x̄_outer, arrondi à l'entier)

def print_snapshots(label, r):
    iters = r['iter_history']
    if not iters:
        print(f"  {label}: aucun snapshot disponible")
        return
    print(f"\n  {label}  (snapshots aux multiples de 5, du plus ancien au plus récent)")
    print(f"  {'t':>8}  {'||g_λ||₂':>12}  {'||x̄_outer||₂':>14}  z*")
    print(f"  {'-'*65}")
    for k, t in enumerate(iters):
        g    = r['g_history'][k]          # shape (N, n) : g[i][k] = x_i*[k] - x̄[k]
        xbar = r['xbar_outer_history'][k] # shape (n,)
        zs   = r['zstar_history'][k]      # shape (N,)
        gnorm  = float(np.linalg.norm(g))
        xbnorm = float(np.linalg.norm(xbar))
        z_str  = '[' + ' '.join(f'{int(z+0.5)}' for z in zs) + ']'
        print(f"  {int(t):>8}  {gnorm:>12.6f}  {xbnorm:>14.6f}  {z_str}")
        # x_ζ_i = g[i] + x̄_outer  (récupère le vecteur x_i* depuis le gradient stocké)
        for i, gi in enumerate(g):
            x_zeta = np.array(gi) + np.array(xbar)
            x_str = '[' + ' '.join(f'{int(v+0.5)}' for v in x_zeta) + ']'
            print(f"  {'':>8}   {'':>12}   {'':>14}  i={i}: x_ζ={x_str}")

for size in SIZES:
    for ep in sorted(RESULTS[size]):
        inst  = DATA[size][ep]['instance']
        v_mip = DATA[size][ep]['v_mip']
        print(f"\n{'='*65}")
        print(f"  {size.upper()} ep={ep}  n={inst.n}, m={inst.m}, N={inst.N}, v*={v_mip:.4f}")
        print(f"{'='*65}")
        print_snapshots("exact ", RESULTS[size][ep]['exact'])
        #print_snapshots("decomp", RESULTS[size][ep]['decomp'])



  SMALL ep=0  n=20, m=5, N=5, v*=4.0000

  exact   (snapshots aux multiples de 5, du plus ancien au plus récent)
         t      ||g_λ||₂   ||x̄_outer||₂  z*
  -----------------------------------------------------------------
      7980      2.898275        1.456022  [1 1 1 1 1]
                                            i=0: x_ζ=[0 0 1 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0]
                                            i=1: x_ζ=[0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0]
                                            i=2: x_ζ=[0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0]
                                            i=3: x_ζ=[0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 1]
                                            i=4: x_ζ=[0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 1 0]
      7985      2.607681        1.280625  [1 1 1 1 1]
                                            i=0: x_ζ=[0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1]
                                            i=1: x_ζ=[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1]


In [5]:
# ── Génération des figures (une par instance) ─────────────────────────────────

def find_plateau_time(best, times):
    """Temps (s) auquel l'amélioration relative < REL_TOL sur la fenêtre."""
    w = max(1, TOL_WINDOW // LOG_INTERVAL)
    for k in range(w, len(best)):
        denom = max(abs(float(best[k])), 1e-10)
        if (float(best[k]) - float(best[k - w])) / denom < REL_TOL:
            return float(times[k])
    return None

PALETTE = {'exact': '#1f77b4', 'decomp': '#ff7f0e'}

saved_files = []
for size in SIZES:
    for ep in sorted(RESULTS[size]):
        v_mip    = DATA[size][ep]['v_mip']
        v_cm_lp  = DATA[size][ep]['v_CM_lp']
        v_cm_ex  = DATA[size][ep]['v_CM_exact']
        inst     = DATA[size][ep]['instance']
        rE       = RESULTS[size][ep]['exact']
        rD       = RESULTS[size][ep]['decomp']

        fig, ax = plt.subplots(figsize=(9, 5))

        # Courbes de convergence
        ax.plot(rE['times_s'], rE['best'],
                color=PALETTE['exact'],  lw=1.8,
                label=f"DP exact   v1LD*={rE['v1LD_star']:.4f}  ({rE['iters']:,} iters)")
        ax.plot(rD['times_s'], rD['best'],
                color=PALETTE['decomp'], lw=1.8,
                label=f"DP décomposé v1LD*={rD['v1LD_star']:.4f}  ({rD['iters']:,} iters)")


        # Référence MIP  (v*)
        if v_mip > 0:
            ax.axhline(v_mip, color='black', linestyle='--', lw=1.2,
                       label=f"v* = {v_mip:.4f} (MIP)")

        # v_C(M) — relaxation LP complète  (x et z continus)
        ax.axhline(v_cm_lp, color='forestgreen', linestyle='-.', lw=1.2,
                   label=f"v_C(M)_LP = {v_cm_lp:.4f}  (LP x,z continus)")

        # v_C(M) exacte Ahmed (x binaire, z continu) — rappel : = v* pour ces instances
        # On l'affiche uniquement si elle diffère de v* (pour ne pas surcharger la légende)
        if abs(v_cm_ex - v_mip) > 1e-4:
            ax.axhline(v_cm_ex, color='purple', linestyle=':', lw=1.0,
                       label=f"v_C(M)_exact = {v_cm_ex:.4f}  (x binaire, z continu)")


        # Marqueurs de plateau (ligne verticale pointillée)
        for key, r, col in [('exact', rE, PALETTE['exact']),
                             ('decomp', rD, PALETTE['decomp'])]:
            t_plat = find_plateau_time(r['best'], r['times_s'])
            if t_plat is not None:
                ax.axvline(t_plat, color=col, linestyle=':', lw=1.0, alpha=0.6,
                           label=f"plateau {key} @ {t_plat:.1f}s")

        ax.set_xlabel('Temps (s)', fontsize=11)
        ax.set_ylabel('Meilleure borne v1LD', fontsize=11)
        ax.set_title(
            f'{size.capitalize()} — épisode {ep}   '
            f'(n={inst.n}, m={inst.m}, N={inst.N}, ε={inst.epsilon})',
            fontsize=12
        )
        ax.legend(fontsize=8.5, loc='lower right')
        ax.grid(True, alpha=0.25)
        plt.tight_layout()

        fname = f"{FIGURE_DIR}/{size}_ep{ep:02d}.jpg"
        fig.savefig(fname, dpi=120, bbox_inches='tight')
        plt.close(fig)
        saved_files.append(fname)
        print(f"  {fname}")

print(f"\n{len(saved_files)} figures sauvegardées.")


  figures/convergence/small_ep00.jpg
  figures/convergence/small_ep01.jpg
  figures/convergence/small_ep02.jpg
  figures/convergence/small_ep03.jpg
  figures/convergence/small_ep04.jpg
  figures/convergence/medium_ep00.jpg
  figures/convergence/medium_ep01.jpg
  figures/convergence/medium_ep02.jpg
  figures/convergence/medium_ep03.jpg
  figures/convergence/medium_ep04.jpg

10 figures sauvegardées.


In [6]:
# ── Tableau récapitulatif ─────────────────────────────────────────────────────
header = (f"{'Taille':>8} {'ep':>3} {'v*':>7} {'v_C(M)_LP':>10} {'v_C(M)_ex':>10} "
          f"{'DP_exact':>10} {'gap_e%':>7} {'iters_e':>10} "
          f"{'DP_decomp':>10} {'gap_d%':>7} {'iters_d':>10} "
)
print(header)
print('-' * len(header))

for size in SIZES:
    for ep in sorted(RESULTS[size]):
        v_mip   = DATA[size][ep]['v_mip']
        v_cm_lp = DATA[size][ep]['v_CM_lp']
        v_cm_ex = DATA[size][ep]['v_CM_exact']
        rE      = RESULTS[size][ep]['exact']
        rD      = RESULTS[size][ep]['decomp']
        gap_e   = (v_mip - rE['v1LD_star']) / v_mip * 100 if v_mip > 0 else float('nan')
        gap_d   = (v_mip - rD['v1LD_star']) / v_mip * 100 if v_mip > 0 else float('nan')
        print(f"{size:>8} {ep:>3} {v_mip:>7.3f} {v_cm_lp:>10.3f} {v_cm_ex:>10.3f} "
              f"{rE['v1LD_star']:>10.4f} {gap_e:>6.1f}% {rE['iters']:>10,} "
              f"{rD['v1LD_star']:>10.4f} {gap_d:>6.1f}% {rD['iters']:>10,} "
)

print()
print("Colonnes :")
print("  v_C(M)_LP    = relaxation LP (x et z continus) — borne inférieure, ≤ v*")
print("  v_C(M)_exact = relaxation z seul (x binaire)   — = v* pour A,b ∈ {0,1}")
print("  gap_e/d %    = (v* - v1LD_star) / v* × 100  (sous-optimalité Lagrangienne)")


  Taille  ep      v*  v_C(M)_LP  v_C(M)_ex   DP_exact  gap_e%    iters_e  DP_decomp  gap_d%    iters_d 
-------------------------------------------------------------------------------------------------------
   small   0   4.000      3.000      4.000     3.9911    0.2%      8,001     3.6842    7.9%     98,001 
   small   1   4.000      3.500      4.000     4.0000    0.0%      2,001     4.0000   -0.0%     31,001 
   small   2   3.000      2.000      3.000     2.3115   22.9%      9,001     2.2884   23.7%    254,001 
   small   3   3.000      2.042      3.000     2.4084   19.7%      6,001     2.3927   20.2%     57,001 
   small   4   4.000      3.125      4.000     4.0000   -0.0%      2,001     4.0000   -0.0%     37,001 
  medium   0   3.000      2.778      3.000     2.7129    9.6%      4,001     2.6590   11.4%    209,001 
  medium   1   3.000      1.681      3.000     1.6450   45.2%      7,001     1.5447   48.5%     94,001 
  medium   2   4.000      2.714      4.000     2.6679   33.3%   

## Diagnostic de convergence — small ep2 & ep3\n\nTrois signaux enregistrés à chaque itération :\n- **Meilleure borne** `best(t)` = max_{s≤t} L1(λ^s)\n- **Norme du sous-gradient** `||g_λ||₂(t)` : grande → on est loin du max ; petite → plateau (plat ou optimum)\n- **Instabilité de z*** : nombre de composantes de z* qui changent entre deux points enregistrés\n\nInterprétation attendue :\n- `||g_λ||` décroît mais reste significatif et `best` stagne → **pas trop petit**, on oscille autour du max sans progresser\n- `||g_λ||` → 0 rapidement et `best` < v* → **la fonction L1 a bien un maximum autour de cette valeur** (sous-optimalité structurelle ?)\n- z* instable → le knapsack n'est pas stabilisé, le sous-gradient reçoit des directions contradictoires

In [7]:
# ── Run diagnostique (grain fin) sur ep2 & ep3 ───────────────────────────────\nDIAG_EPISODES  = [2, 3]\nDIAG_TIME_S    = 120        # 2 min par run\nDIAG_LOG       = 10         # enregistrer toutes les 10 itérations\n\nDIAG = {}\nfor ep in DIAG_EPISODES:\n    inst   = DATA['small'][ep]['instance']\n    v_mip  = DATA['small'][ep]['v_mip']\n    print(f\"\\n[small ep={ep}] exact — {DIAG_TIME_S}s ... \", end='', flush=True)\n\n    res = v1ld_cpp2.solve(\n        inst.c.astype(float), inst.p.astype(float),\n        inst.A.astype(float), inst.b.astype(float),\n        inst.epsilon,\n        T=T_MAX,\n        decomp=False,\n        verbose=False,\n        time_limit_s=DIAG_TIME_S,\n        log_interval=DIAG_LOG,\n        rel_tol=-1.0,\n        tol_window=500,\n    )\n\n    conv    = np.array(res['convergence'])\n    times   = np.array(res['times_s'])\n    best    = np.maximum.accumulate(conv)\n    gnorm   = np.array(res['gnorm_history'])\n    zchg    = np.array(res['z_changes'])\n\n    DIAG[ep] = {\n        'conv': conv, 'best': best, 'times': times,\n        'gnorm': gnorm, 'zchg': zchg,\n        'v_mip': v_mip, 'v1LD_star': res['v1LD_star'],\n        'iters': res['iterations_run'],\n    }\n    print(f\"{res['iterations_run']:,} iters  v1LD*={res['v1LD_star']:.6f}  v*={v_mip:.4f}\")\n\nprint(\"\\nDiagnostic terminé.\")

In [8]:
# ── Figures diagnostiques ────────────────────────────────────────────────────\nfor ep in DIAG_EPISODES:\n    d     = DIAG[ep]\n    v_mip = d['v_mip']\n    t     = d['times']\n\n    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))\n    fig.suptitle(\n        f'Small ep{ep} — Diagnostic sous-gradient  '\n        f'(n={DATA[\"small\"][ep][\"instance\"].n}, '\n        f'N={DATA[\"small\"][ep][\"instance\"].N}, v*={v_mip:.4f})',\n        fontsize=12\n    )\n\n    # ── (1) Meilleure borne ────────────────────────────────────────────────\n    ax = axes[0]\n    ax.plot(t, d['best'], color='#1f77b4', lw=1.5)\n    ax.axhline(v_mip, color='crimson', ls='--', lw=1.2, label=f'v* = {v_mip:.4f}')\n    ax.axhline(d['v1LD_star'], color='grey', ls=':', lw=1.0,\n               label=f\"v1LD* = {d['v1LD_star']:.4f}\")\n    ax.set_xlabel('Temps (s)')\n    ax.set_ylabel('best L1(λ)')\n    ax.set_title('Meilleure borne')\n    ax.legend(fontsize=8)\n    ax.grid(True, alpha=0.25)\n\n    # ── (2) Norme du sous-gradient ─────────────────────────────────────────\n    ax = axes[1]\n    ax.semilogy(t, d['gnorm'], color='darkorange', lw=1.2, alpha=0.7)\n    # Moyenne mobile (fenêtre 5% des points)\n    w = max(1, len(d['gnorm']) // 20)\n    if len(d['gnorm']) > w:\n        smooth = np.convolve(d['gnorm'], np.ones(w)/w, mode='valid')\n        t_smooth = t[:len(smooth)]\n        ax.semilogy(t_smooth, smooth, color='darkred', lw=2.0, label=f'moyenne {w*DIAG_LOG} iters')\n    ax.set_xlabel('Temps (s)')\n    ax.set_ylabel('||g_λ||₂  (log)')\n    ax.set_title('Norme sous-gradient')\n    ax.legend(fontsize=8)\n    ax.grid(True, alpha=0.25)\n\n    # ── (3) Instabilité z* ─────────────────────────────────────────────────\n    ax = axes[2]\n    # Proportion de runs où z* change (moyenne mobile)\n    w2 = max(1, len(d['zchg']) // 30)\n    if len(d['zchg']) > w2:\n        frac = np.convolve(d['zchg'] > 0, np.ones(w2)/w2, mode='valid')\n        t_z  = t[:len(frac)]\n        ax.plot(t_z, frac, color='green', lw=1.8)\n    else:\n        ax.plot(t[:len(d['zchg'])], (d['zchg'] > 0).astype(float), color='green', lw=1.0)\n    ax.set_ylim(-0.05, 1.05)\n    ax.set_xlabel('Temps (s)')\n    ax.set_ylabel('Fraction de pas où z* change')\n    ax.set_title(f'Instabilité z* (fenêtre {w2*DIAG_LOG} iters)')\n    ax.grid(True, alpha=0.25)\n\n    plt.tight_layout()\n    fname = f\"{FIGURE_DIR}/small_ep{ep:02d}_diagnostic.jpg\"\n    fig.savefig(fname, dpi=120, bbox_inches='tight')\n    plt.close(fig)\n    print(fname)